# IMPORT
---

In [ ]:
!pip install optuna

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import confusion_matrix, accuracy_score
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# DATASET
---

In [ ]:
url = "https://zenodo.org/records/5724362/files/PEMS-BAY.csv"
df = pd.read_csv(url)

PROJECT_DIR = "/content/drive/MyDrive/Traffic_Flow_LSTM"
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
MODEL_DIR = os.path.join(PROJECT_DIR, "model")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0])
df.set_index(df.columns[0], inplace=True)

print("Dataset shape:", df.shape)

In [ ]:
df.replace(0, np.nan, inplace=True)
df.interpolate(method="time", inplace=True)
df = df.bfill().ffill()

print("Missing values:", df.isna().sum().sum())

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df.values)

Missing values: 0


In [ ]:
def create_sequences(data, window):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i:i + window])
        y.append(data[i + window])
    return np.array(X), np.array(y)


# MODEL
---

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc1 = nn.Linear(hidden_size, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, input_size)

    def forward(self, x):

        out, _ = self.lstm(x)

        out = out[:, -1, :]
        out = self.relu(self.fc1(out))
        out = self.fc2(out)
        return out


# TRAIN
---

In [ ]:
def objective(trial):
    window_size = trial.suggest_int("window_size", 6, 24)
    hidden_size = trial.suggest_int("lstm_units", 32, 128)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    lr = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    X, y = create_sequences(scaled_data, window_size)

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, shuffle=False
    )

    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_val = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val = torch.tensor(y_val, dtype=torch.float32).to(device)

    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=False
    )

    val_loader = DataLoader(
        TensorDataset(X_val, y_val),
        batch_size=batch_size,
        shuffle=False
    )

    model = LSTMModel(input_size=X.shape[2], hidden_size=hidden_size).to(device)

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for _ in range(10):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            val_loss += criterion(model(xb), yb).item()

    return val_loss / len(val_loader)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

study = optuna.create_study(direction="minimize")

study.optimize(objective, n_trials=20)

print("\nOPTUNA BEST PARAMETERS")
print("Best MSE:", study.best_value)
print("Best Params:", study.best_params)

In [ ]:
best = study.best_params

X, y = create_sequences(scaled_data, best["window_size"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
X_test = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).to(device)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=best["batch_size"],
    shuffle=False
)

model = LSTMModel(
    input_size=X.shape[2],
    hidden_size=best["lstm_units"]
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=best["learning_rate"]
)

In [ ]:
for epoch in range(30):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()


# EVALUATION
---

In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(X_test).cpu().numpy()

y_test_np = y_test.cpu().numpy()

mse = mean_squared_error(y_test_np.flatten(), y_pred.flatten())
mae = mean_absolute_error(y_test_np.flatten(), y_pred.flatten())
rmse = np.sqrt(mse)

print("\nREGRESSION METRICS")
print("MSE :", mse)
print("MAE :", mae)
print("RMSE:", rmse)


In [ ]:
metrics_csv_path = os.path.join(RESULTS_DIR, "metrics.csv")

metrics_df = pd.DataFrame({
    "Metric": ["MSE", "MAE", "RMSE"],
    "Value": [mse, mae, rmse]
})

metrics_df.to_csv(metrics_csv_path, index=False)
print(f"Metrics saved to CSV at: {metrics_csv_path}")

metrics_path = os.path.join(RESULTS_DIR, "metrics.txt")

with open(metrics_path, "w") as f:
    f.write("Regression Metrics (Test Set)\n")
    f.write(f"MSE : {mse}\n")
    f.write(f"MAE : {mae}\n")
    f.write(f"RMSE: {rmse}\n")

print(f"Metrics saved to {metrics_path}")

In [ ]:
sensor_idx = 0

plt.figure(figsize=(12, 4))
plt.plot(y_test_np[:300, sensor_idx], label="True")
plt.plot(y_pred[:300, sensor_idx], label="Predicted")
plt.title("Optuna-Tuned LSTM Traffic Prediction (Sensor 0)")
plt.legend()

plot_path = os.path.join(RESULTS_DIR, "prediction_sensor0.png")
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"Prediction plot saved to {plot_path}")

In [ ]:
def traffic_class(x):
    if x < 0.33:
        return 0
    elif x < 0.66:
        return 1
    else:
        return 2

y_true_cls = np.vectorize(traffic_class)(y_test_np)
y_pred_cls = np.vectorize(traffic_class)(y_pred)

cm = confusion_matrix(y_true_cls.flatten(), y_pred_cls.flatten())
acc = accuracy_score(y_true_cls.flatten(), y_pred_cls.flatten())

print("\nCLASSIFICATION METRICS")
print("Accuracy:", acc)
print("Confusion Matrix:\n", cm)
